<a href="https://colab.research.google.com/github/codex319/SIH-26043-Complaint-Dataset/blob/main/Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:

# SIH 26043 - COMPLETE DATA PREPROCESSING


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,SimpleRNN,LSTM,Embedding,Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')
import re
import nltk
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder



# 1. UPLOAD YOUR ORIGINAL TXT DATASET


print("Please upload your original .txt dataset:")

input_file = "sih26043_complaints_dataset.txt"

print("\nUploaded file:", input_file)



# 2. DOWNLOAD ENGLISH STOPWORDS


nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))

print("Stopwords loaded:", len(stop_words))



# 3. READ DATASET


df = pd.read_csv(
    input_file,
    sep=";",
    encoding="utf-8"
)

print("\nOriginal columns:")
print(df.columns.tolist())

print("\nOriginal shape:")
print(df.shape)



# 4. TEXT CLEANING FUNCTION
#    - Lowercase
#    - Punctuation removal
#    - Stopword removal


def clean_text(text):

    # Convert to string
    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = re.sub(r"[^\w\s]", " ", text)

    # Split into words
    words = text.split()

    # Remove English stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Remove extra spaces
    text = " ".join(words)

    return text



# 5. APPLY TEXT CLEANING



df["cleaned_complaint"] = df["complaint"].apply(clean_text)




# 6. LABEL ENCODING
#    Category -> Numerical Label



label_encoder = LabelEncoder()

df["category_label"] = label_encoder.fit_transform(
    df["category"]
)



# 7. SHOW CATEGORY-LABEL MAPPING


print("\n========== CATEGORY LABEL MAPPING ==========")

for category, label in zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
):
    print(f"{category}  ->  {label}")


# 8. CHECK CLEANING


print("\n========== ORIGINAL VS CLEANED ==========")

print(
    df[["complaint", "cleaned_complaint"]]
    .head(10)
    .to_string(index=False)
)



# 9. CREATE FINAL DATASET


final_df = df[
    [
        "cleaned_complaint",
        "category",
        "category_label"
    ]
]



# 10. SAVE AS TXT


final_df.to_csv(
    "final_dataset.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)


# 11. FINAL CHECK


print("\n========== FINAL DATASET ==========")

print(final_df.head(10).to_string(index=False))

print("\nFinal shape:", final_df.shape)

print("\nFinal columns:")
print(final_df.columns.tolist())

print("\nFile created successfully:")
print(output_file)


# 12. DOWNLOAD FINAL TXT FILE

print("\n========== DONE ==========")
print("Your preprocessed TXT dataset has been downloaded.")

Please upload your original .txt dataset:

Uploaded file: sih26043_complaints_dataset.txt
Stopwords loaded: 198

Original columns:
['complaint', 'category']

Original shape:
(30000, 2)

========== CATEGORY LABEL MAPPING ==========
Agriculture & Rural Development  ->  0
Air & Environmental Pollution  ->  1
Education  ->  2
Energy & Sustainability  ->  3
Infrastructure & Roads  ->  4
Public Health  ->  5
Safety & Public Welfare  ->  6
Waste Management  ->  7
Water Quality & Sanitation  ->  8

========== ORIGINAL VS CLEANED ==========
                                                                                                                                                      complaint                                                                                            cleaned_complaint
                                                                     Hello, the road divider near Bero market is broken and causing traffic problems every day.                                he

In [57]:
final_df


,cleaned_complaint,category,category_label
0,hello road divider near bero market broken cau...,Infrastructure & Roads,4
1,may concern people dumping waste open plot nea...,Waste Management,7
2,resident ramgarh several incidents chain snatc...,Safety & Public Welfare,6
3,hello streetlight road near nagla basti 2 mont...,Infrastructure & Roads,4
4,child unwell health worker visited village vac...,Public Health,5
...,...,...,...
29995,respected sir madam sewage water mixing drinki...,Water Quality & Sanitation,8
29996,sir municipal waste truck skips area days kind...,Waste Management,7
29997,farmers pathalgaon want inform received soil t...,Agriculture & Rural Development,0
29998,sir ji open manholes devgarh road risky night ...,Infrastructure & Roads,4


In [58]:


# Step 1: get plain text and labels as separate flat lists/arrays
X = df['cleaned_complaint'].tolist()      # list of complaint strings
y = df['category_label'].values           # array of numeric labels

# Step 2: build a tokenizer that converts words to integer IDs
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)                 # learns the vocabulary from your complaints

# Step 3: convert each complaint sentence into a sequence of integers (one per word)
sequences = tokenizer.texts_to_sequences(X)

# Step 4: NOW compute maxlen — on the tokenized sequences, not raw X
maxlen = max(len(seq) for seq in sequences)
print(maxlen)   # this should now print something realistic, e.g. 15-30 (word count of longest complaint)

# Step 5: pad every sequence to the same length so they can be fed into the LSTM as a batch
X_padded = pad_sequences(sequences, maxlen=maxlen, padding='post')

26


In [59]:
X_padded.shape

(30000, 26)

In [60]:
vocab_size = len(tokenizer.word_index) + 1 
embedding_dim = 64
lstm_units = 64

In [61]:
X_train, X_val, y_train, y_val = train_test_split(
    X_padded, y,
    test_size=0.1,
    random_state=42,
    stratify=y       
)

In [62]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=maxlen),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(9, activation='softmax')   # 9 output classes, one per category
])

In [63]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [64]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [65]:
lstm_history=model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64
)

Epoch 1/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - accuracy: 0.7031 - loss: 0.7462 - val_accuracy: 1.0000 - val_loss: 0.0053
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9900 - loss: 0.0483 - val_accuracy: 1.0000 - val_loss: 5.3606e-04
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 18s 44ms/step - accuracy: 0.9950 - loss: 0.0259 - val_accuracy: 1.0000 - val_loss: 1.1177e-04
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 19s 44ms/step - accuracy: 0.9974 - loss: 0.0148 - val_accuracy: 1.0000 - val_loss: 3.1626e-05
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 48ms/step - accuracy: 0.9977 - loss: 0.0117 - val_accuracy: 1.0000 - val_loss: 1.2774e-05
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.9990 - loss: 0.0061 - val_accuracy: 1.0000 - val_loss: 5.1325e-06
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 19s 44ms/step - accuracy: 0.9991 - loss: 0.0044 - val_accuracy: 1.0000 - val_loss: 2.1466e-06
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 

In [68]:
# when you encoded your 9 category names into numbers for training
label_map = dict(enumerate(df['category'].astype('category').cat.categories))
print(label_map)

{0: 'Agriculture & Rural Development', 1: 'Air & Environmental Pollution', 2: 'Education', 3: 'Energy & Sustainability', 4: 'Infrastructure & Roads', 5: 'Public Health', 6: 'Safety & Public Welfare', 7: 'Waste Management', 8: 'Water Quality & Sanitation'}


In [66]:
model.save('complaint_lstm_model.keras')

In [69]:
import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('label_map.pkl', 'wb') as f:
    pickle.dump(label_map, f)

In [ ]:
import kagglehub

path1 = kagglehub.dataset_download("emmarex/plantdisease")
path2 = kagglehub.dataset_download("mostafaabla/garbage-classification")
path3 = kagglehub.dataset_download("adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal")
path4 = kagglehub.dataset_download("kabeer2004/water-pollution-images")
path5 = kagglehub.dataset_download("akinduhiman/urban-issues-dataset")
path6 = kagglehub.dataset_download("alicjalena/pv-panel-defect-dataset")

print(path1)
print(path2)
print(path3)
print(path4)
print(path5)
print(path6)


Resuming download from 30408704 bytes (636705064 bytes left)...
Resuming download to C:\Users\bless\.cache\kagglehub\datasets\adarshrouniyar\air-pollution-image-dataset-from-india-and-nepal\10.archive (30408704/667113768) bytes left.


 60%|█████▉    | 380M/636M [06:09<04:19, 1.04MB/s] 